In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor


device = "cuda"

In [ ]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split

nmt_original_valid_set, nmt_test_set = load_dataset(
    path="ageron/tatoeba_mt_train", name="eng-spa", split=["validation", "test"]
) 
split = nmt_original_valid_set.train_test_split(train_size=0.9, seed=42)
nmt_train_set, nmt_test_set = split["train"], split["test"]

In [3]:
nmt_train_set[0]

{'source_text': 'The two teams debated on the issue of nuclear power.',
 'target_text': 'Los dos equipos debatieron sobre el tema de la energía nuclear.',
 'source_lang': 'eng',
 'target_lang': 'spa'}

In [ ]:
import tokenizers
import tokenizers.pre_tokenizers
import tokenizers.trainers
from collections import namedtuple
from torch.utils.data import DataLoader

def get_data_pair():
    for pair in nmt_train_set:
        yield pair["source_text"]
        yield pair["target_text"]

max_length = 256
vocab_size = 10000
nmt_tokenizer = tokenizers.Tokenizer(tokenizers.models.BPE(unk_token="<unk>"))
nmt_tokenizer.enable_padding(pad_id=0, pad_token="<pad>")
nmt_tokenizer.enable_truncation(max_length=max_length)
nmt_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()
nmt_tokenizer_trainer = tokenizers.trainers.BpeTrainer(
    vocab_size=vocab_size, special_tokens=["<pad>", "<unk>", "<s>", "</s>"]
)
nmt_tokenizer.train_from_iterator(get_data_pair(), nmt_tokenizer_trainer)

In [ ]:
from torch.utils.data import DataLoader


fields = ["src_token_ids", "src_mask", "tgt_token_ids", "tgt_mask"]
class NmtPair(namedtuple("NmtPairBase", fields)):
    def to(self, device):
        return NmtPair(*[elem.to(device) for elem in self])

def nmt_collate_fn(batch):
    src_texts = [pair['source_text'] for pair in batch]
    tgt_texts = [f"<s> {pair['target_text']} </s>" for pair in batch]
    src_encodings = nmt_tokenizer.encode_batch(src_texts)
    tgt_encodings = nmt_tokenizer.encode_batch(tgt_texts)
    src_token_ids = torch.tensor([enc.ids for enc in src_encodings])
    tgt_token_ids = torch.tensor([enc.ids for enc in tgt_encodings])
    src_mask = torch.tensor([enc.attention_mask for enc in src_encodings])
    tgt_mask = torch.tensor([enc.attention_mask for enc in tgt_encodings])
    inputs = NmtPair(src_token_ids, src_mask, tgt_token_ids[:,:-1], tgt_mask[:,:-1])
    labels = tgt_token_ids[:,1:]
    return inputs, labels

batch_size=32
pin_memory=True
nmt_train_loader = DataLoader(nmt_train_set, batch_size, shuffle=True, collate_fn=nmt_collate_fn, pin_memory=pin_memory)
nmt_test_loader = DataLoader(nmt_test_set, batch_size, shuffle=True, collate_fn=nmt_collate_fn, pin_memory=pin_memory) 

In [ ]:
from typing import Any, Optional

class PosEmbedding(nn.Module):
    def __init__(self, max_length:int, embed_dim:int, dropout:float=0.1):
        super().__init__()
        self.pos_embed = nn.Parameter(torch.randn(max_length, embed_dim) * 0.02)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, X):
        return self.dropout(X+self.pos_embed[:X.size(1)])

class MultiheadAttention(nn.Module):
    def __init__(self, embed_dim:int, num_heads:int, dropout:float=0.1):
        super().__init__()
        self.num_heads:int = num_heads
        self.head_dim = embed_dim//num_heads
        self.query = nn.Linear(embed_dim, embed_dim)
        self.key = nn.Linear(embed_dim, embed_dim)
        self.value = nn.Linear(embed_dim, embed_dim)
        self.output = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)
    
    def split_head(self, X:Tensor):
        return X.view(X.size(0), X.size(1), self.num_heads, self.head_dim).transpose(1,2)

    def forward(self, query:Tensor, key:Tensor, value:Tensor, 
                attn_mask:Optional[Tensor]=None, 
                key_padding_mask:Optional[Tensor]=None) ->tuple[Tensor, Tensor]:
        q = self.split_head(self.query(query))
        k = self.split_head(self.key(key))
        v = self.split_head(self.value(value))
        
        scores_mat = q @ k.transpose(2,3)/self.head_dim**0.5 #transpose for dot product
        if attn_mask:
            scores_mat = scores_mat.masked_fill(attn_mask, -torch.inf)
        if key_padding_mask:
            mask = key_padding_mask.unsqueeze(1).unsqueeze(2)
            scores_mat = scores_mat.masked_fill(mask, -torch.inf)
        
        scores = scores_mat.softmax(dim=-1)
        scores = self.dropout(scores) #(B, h, Lq, d)
        
        Z = scores @ v
        Z = Z.transpose(1,2) # (B, Lq, h, d)
        Z = Z.reshape(Z.size(0), Z.size(1), Z.size(2) * Z.size(3))
        
        return (self.output(Z), scores)
    
class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model:int, num_heads:int, dim_feedforward:int=2048, dropout:float=0.1) -> None:
        super().__init__()
        self.self_attn = MultiheadAttention(d_model, num_heads, dropout)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
    
    def forward(self, src:Tensor, src_mask:Optional[Tensor]=None, key_padding_mask:Optional[Tensor]=None):
        attention, _ = self.self_attn(src, src, src, attn_mask=src_mask, key_padding_mask=key_padding_mask)
        attention_final = self.norm1(src + self.dropout(attention))
        
        feedforward_in = self.dropout(self.linear1(attention_final).relu())
        feedforward_out = self.dropout(self.linear2(feedforward_in))
        
        return self.norm2(attention_final + feedforward_out) # <- memory
    
class TransformerDecoderLayer(nn.Module):
    def __init__(self, d_model:int, num_heads:int, dim_feedforward:int=2048, dropout:float=0.1):
        super().__init__()
        
        self.self_atten = MultiheadAttention(d_model, num_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        
        self.cross_atten = MultiheadAttention(d_model, num_heads, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        
        self.ff_input = nn.Linear(d_model, dim_feedforward)
        self.ff_output = nn.Linear(dim_feedforward, d_model)
        self.norm3 = nn.LayerNorm(d_model)        
        self.dropout = nn.Dropout(dropout)
            
    def forward(self, tgt:Tensor, memory:Tensor, 
                tgt_mask:Optional[Tensor] = None, memory_mask:Optional[Tensor]=None, 
                tgt_key_padding:Optional[Tensor]=None, memory_key_padding_mask:Optional[Tensor]=None):
        self_attention, _ = self.self_atten(tgt, tgt, tgt, attn_mask=tgt_mask, key_padding_mask=tgt_key_padding)        
        q = self.norm1(tgt + self.dropout(self_attention))
        
        cross_attention, _ = self.cross_atten(q, memory, memory, attn_mask=memory_mask, key_padding_mask=memory_key_padding_mask)
        cross_attention_block_output = self.norm2(q + self.dropout(cross_attention))
        
        ff_input_result = self.dropout(self.ff_input(cross_attention_block_output).relu())
        ff_output_result = self.dropout(self.ff_output(ff_input_result))
        ff_result = self.norm3(cross_attention_block_output + ff_output_result)
        
        return ff_result

In [ ]:
class NmtTransformer(nn.Module):
    def __init__(self, vocab_size:int, max_length:int, embed_dim:int=512, 
                 pad_id:int=0, num_heads:int=8, num_layers:int=6, dropout:float=0.1):
        super().__init__()
        self.num_layers = num_layers
        self.embed = nn.Embedding(vocab_size, embed_dim, pad_id)
        self.pos_embed = PosEmbedding(max_length, embed_dim, dropout)
        self.encoder = TransformerEncoderLayer(embed_dim, num_heads, dropout=dropout)
        self.decoder = TransformerDecoderLayer(embed_dim, num_heads, dropout=dropout)
        self.output = nn.Linear(embed_dim, vocab_size)
    
    def forward(self, pair):
        src_embeds = self.pos_embed(self.embed(pair.src_token_ids))
        tgt_embeds = self.pos_embed(self.embed(pair.tgt_token_ids))
        src_pad_mask = ~pair.src_mask.bool() #in pytorch, True means masking
        tgt_pad_mask = ~pair.tgt_mask.bool()
        size = [pair.tgt_token_ids.size(1)] * 2
        full_mask = torch.full(size, True, device=tgt_pad_mask.device)
        causal_mask = torch.triu(full_mask, diagonal=1)
        
        
        memory = src_embeds
        for encoder_layer in range(self.num_layers):
            memory = encoder_layer(src=memory, key_padding_mask=src_pad_mask)
        
        decoder_output = tgt_embeds
        for decoder_layer in range(self.num_layers):
            decoder_output = self.decoder(tgt=tgt_embeds, memory=memory, tgt_mask=causal_mask, 
                                      tgt_key_padding=tgt_pad_mask, memory_key_padding_mask=src_pad_mask)
        
        return self.output(decoder_output).permute(0,2,1)

In [ ]:
from torch.utils.data import DataLoader

model = NmtTransformer(vocab_size, max_length, embed_dim=128, pad_id=0, num_heads=4, num_layers=2, dropout=0.1)
optimizer = torch.optim.NAdam(model.parameters(), lr=0.002)
xentropy = nn.CrossEntropyLoss(ignore_index=0)

def train_epoch(model:nn.Module, loader:Dataloader, loss_fn:nn.Module, optimizer:torch.optim.Optimizer):
    model.train()
    total_loss = 0
    for inputs, labels in loader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        logits = model(inputs)
        loss = loss_fn(logits,labels)
        loss.backward()
        optimizer.step()
        total_loss = loss.item()
    
    return total_loss/len(loader)

def eval_epoch(model:nn.Module, loader:DataLoader, loss_fn:nn.Module):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            logits = model(inputs)
            loss = loss_fn(logits, labels)
            total_loss += loss.item()
    
    return total_loss / len(loader)


def translate(model, src_text, max_length=20, pad_id=0, eos_id=3):
    tgt_text = ""
    output_ids = []
    for index in range(max_length):
        batch, _ = nmt_collate_fn([{"source_text": src_text, "target_text": tgt_text}])
        with torch.no_grad():
            Y_logits = model(batch.to(device))
            Y_token_ids = Y_logits.argmax(dim=1)
            next_token_id = Y_token_ids[0, index].item()

        if next_token_id == eos_id:
            break
        output_ids.append(next_token_id)
        next_token = nmt_tokenizer.id_to_token(next_token_id)
        tgt_text += " " + next_token

    return nmt_tokenizer.decode(output_ids)